In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
create widget text storageName default "adlsssmartdataanayeli2";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-catalog-mt`
URL 'abfss://catalog-mt@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
DROP CATALOG IF EXISTS proyecto_olist CASCADE;

In [0]:
CREATE CATALOG IF NOT EXISTS proyecto_olist
MANAGED LOCATION 'abfss://catalog-mt@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

In [0]:
DROP SCHEMA IF EXISTS proyecto_olist.raw;
DROP SCHEMA IF EXISTS proyecto_olist.bronze;
DROP SCHEMA IF EXISTS proyecto_olist.silver;
DROP SCHEMA IF EXISTS proyecto_olist.golden;

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/",True)

In [0]:
CREATE SCHEMA IF NOT EXISTS proyecto_olist.raw;
CREATE SCHEMA IF NOT EXISTS proyecto_olist.bronze;
CREATE SCHEMA IF NOT EXISTS proyecto_olist.silver;
CREATE SCHEMA IF NOT EXISTS proyecto_olist.golden;

###Tablas Bronze

In [0]:
CREATE TABLE IF NOT EXISTS proyecto_olist.bronze.customers (
    customer_id STRING,
    customer_unique_id STRING,
    customer_zip_code_prefix INT,
    customer_city STRING,
    customer_state STRING,
    ingestion_date TIMESTAMP,
    source_file STRING
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/customers"

In [0]:
CREATE TABLE IF NOT EXISTS proyecto_olist.bronze.orders (
    order_id STRING,
    customer_id STRING,
    order_status STRING,
    order_purchase_timestamp TIMESTAMP,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    ingestion_date TIMESTAMP,
    source_file STRING,
    processing_date DATE
)
USING DELTA
PARTITIONED BY (processing_date)
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/orders"

In [0]:
CREATE TABLE IF NOT EXISTS proyecto_olist.bronze.products (
    product_id STRING,
    product_category_name STRING,
    product_name_lenght INT,
    product_description_lenght INT,
    product_photos_qty INT,
    product_weight_g DOUBLE,
    product_length_cm DOUBLE,
    product_height_cm DOUBLE,
    product_width_cm DOUBLE,
    ingestion_date TIMESTAMP,
    source_file STRING
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/products"

In [0]:
CREATE TABLE IF NOT EXISTS proyecto_olist.bronze.order_items (
    order_id STRING,
    order_item_id INT,
    product_id STRING,
    seller_id STRING,
    shipping_limit_date TIMESTAMP,
    price DOUBLE,
    freight_value DOUBLE,
    ingestion_date TIMESTAMP,
    source_file STRING
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/order_items"

###Tablas Silver

In [0]:
CREATE TABLE IF NOT EXISTS proyecto_olist.silver.customers (
    customer_id STRING,
    customer_unique_id STRING,
    customer_zip_code_prefix INT,
    customer_city STRING,
    customer_state STRING,
    silver_insert_date TIMESTAMP
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/customers"

In [0]:
CREATE TABLE IF NOT EXISTS proyecto_olist.silver.orders (
    order_id STRING,
    customer_id STRING,
    order_status STRING,
    order_purchase_timestamp TIMESTAMP,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    silver_insert_date TIMESTAMP
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/orders"

In [0]:
CREATE TABLE IF NOT EXISTS proyecto_olist.silver.products (
    product_id STRING,
    product_category_name STRING,
    product_name_lenght INT,
    product_description_lenght INT,
    product_photos_qty INT,
    product_weight_g DOUBLE,
    product_length_cm DOUBLE,
    product_height_cm DOUBLE,
    product_width_cm DOUBLE,
    silver_insert_date TIMESTAMP
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/products"

In [0]:
CREATE TABLE IF NOT EXISTS proyecto_olist.silver.order_items (
    order_id STRING,
    order_item_id INT,
    product_id STRING,
    seller_id STRING,
    shipping_limit_date TIMESTAMP,
    price DOUBLE,
    freight_value DOUBLE,
    silver_insert_date TIMESTAMP
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/order_items"

###Tablas Golden

In [0]:
CREATE TABLE IF NOT EXISTS proyecto_olist.golden.dim_customers (
    customer_id STRING,
    customer_unique_id STRING,
    customer_city STRING,
    customer_state STRING,
    gold_insert_date TIMESTAMP
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/dim_customers"

In [0]:
CREATE TABLE IF NOT EXISTS proyecto_olist.golden.dim_products (
    product_id STRING,
    product_category_name STRING,
    product_weight_g DOUBLE,
    product_length_cm DOUBLE,
    product_height_cm DOUBLE,
    product_width_cm DOUBLE,
    gold_insert_date TIMESTAMP
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/dim_products"

In [0]:
CREATE TABLE IF NOT EXISTS proyecto_olist.golden.fact_orders (
    order_id STRING,
    customer_id STRING,
    product_id STRING,
    order_status STRING,
    order_purchase_timestamp TIMESTAMP,
    amount DOUBLE,
    shipping_cost DOUBLE,
    gold_insert_date TIMESTAMP
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/fact_orders"

In [0]:
SHOW TABLES IN proyecto_olist.bronze;
SHOW TABLES IN proyecto_olist.silver;
SHOW TABLES IN proyecto_olist.golden;